# Testing and debugging the dataset creation

In [17]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
import time
import sys
sys.path.append("..")
import importlib
import scripts.ingest

importlib.reload(scripts.ingest)

from scripts.ingest import (
    processed_date,
    scrape_archive_metadata,
    selection_logic,
    extract_slug,
    scrape_edition,
    filter_parts,
    clean_q4_and_answer,
    url_headline_extract,
    process_edition,
    append_and_store,
    check_existence_process_append_store
)

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

client = Firecrawl(api_key=api_key)

In [7]:
full_archive_metadata = scrape_archive_metadata(client)

In [8]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

data_path = "../data"

index = 2054
edition_metadata = full_archive_metadata[index]

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
print(date)
edition_metadata

2020-10-31


{'_id': '6NCWJjjF63C8yiPtsZW3pRx6',
 'date': '2020-10-31T10:00:00.000+00:00',
 'previewText': "The Halloween Issue you've been waiting for...",
 'slug': '2020/10/31',
 'subjectLine': 'Boo',
 'title': 'The Halloween Issue ',
 'websitePreviewText': None}

In [9]:
selection_logic(date)

True

In [10]:
slug = extract_slug(edition_metadata, date)
print(slug)

2020%2F10%2F31


In [13]:
scraped_edition = scrape_edition(client, slug)
scraped_edition.markdown

'[Skip to main content](https://www.morningbrew.com/issues/2020%2F10%2F31#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Boo\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| The Halloween Issue you\'ve been waiting for... |\n\n{beacon}\n\n| | |     |     |\n| --- | --- |\n| October 31, 2020 | [View Online](https://www.morningbrew.com/daily/iss

In [ ]:
split_quiz = scraped_edition.markdown.split("<br>")
split_quiz

In [18]:
questions_and_answers = filter_parts(scraped_edition)
questions_and_answers

["1. Joe Exotic's 'Tiger King’ zoo is [rumored to be haunted](https://www.newsweek.com/joe-exotics-tiger-king-zoo-haunted-ghost-adventures-wants-prove-it-1543248?utm_source=morning_brew) and ‘Ghost Adventures’ wants to prove it",
 '2. Beloved chain Fatburger makes its [Dallas debut](https://dallas.eater.com/2020/10/27/21534599/fat-brands-dallas-ghost-kitchen-open-fatburger-cafe-momentum-gapco-pizza-the-stand-opening?utm_source=morning_brew) via a ghost kitchen',
 "3. Tinder's [new feature](https://media.giphy.com/media/Rkt9N7lIm1IsKnh3tZ/giphy.gif?utm_source=morning_brew) warns users when they're about to be 'ghosted'",
 "4. Dunkin' adds [Spicy Ghost Pepper Donut](https://www.foxnews.com/food-drink/dunkin-spicy-ghost-pepper-donut-halloween-offerings?utm_source=morning_brew) to traditional Halloween offerings | |\n| --- |\n\n| |     |\n| --- |\n| ### SHARE THE BREW |\n\n|     |\n| --- |\n| {if profile.vars.referral\\_count == 0}"]

In [19]:
questions_and_answers = clean_q4_and_answer(questions_and_answers)
questions_and_answers

IndexError: list index out of range

In [16]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
part = '# [Demon Stores Look for New Hosts](https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020?email={{profile.email}}) |\n\n![Spirit Halloween store](https://dlp31coh2a67q.cloudfront.net/eyJrZXkiOiJ1cGxvYWRzL21lZGl1bS9hc3NldC8yOTkwL3NwaXJpdF9oYWxsb3dlZW5fc3RvcmUuanBnIiwiYnVja2V0Ijoib3Nsby1wcm9kdWN0aW9uIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoxMDAwLCJoZWlnaHQiOm51bGx9fX0=)\n\n_Ben McCanna/Portland Portland Press Herald via Getty Images_\n\n|     |\n| --- |\n| Every year, pop-up Halloween stores rise from the underworld to possess empty storefronts vacated by mortal retailers.\n\nTypically, this non-traditional leasing model means seasonal pop-ups perform well when the rest of retail flounders. But even with 2020 laying out a feast of fresh retail corpses, are spooky stores immune to the pandemic? Let\'s check in with two retail reanimaters:\n\n1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+) than last year, and its CEO said last month the company is "anticipating a Halloween on par with last year."\n2. **Party City** was far less bullish, opening 91% fewer of its Halloween City locations this year. With just 25 stores across the U.S., these days it\'s more like a Halloween Town.\n\n**Zoom out:** Halloween City was showing zombie-like symptoms even before the pandemic. Last year, sales per store declined nearly [21% annually](https://www.retaildive.com/news/party-citys-october-sales-a-disaster-with-halloween-stores-down-21/566955/?utm_source=morning_brew)—a major problem for parent Party City considering Halloween sales account for a fifth of its total U.S. retail revenue.\n\n|     |\n| --- |\n| [![](https://media.sailthru.com/5z8/1k4/4/2/5e865734bf6f7.jpg)](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86574d1eb82.jpg)](https://twitter.com/intent/tweet?text=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year&url=https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020&via=MorningBrew)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86576e04837.jpg)](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020&title=Halloween+Pop-Up+Shops+Have+Mixed+Success+In+2020&source=www.morningbrew.com&summary=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year)[![](https://media.sailthru.com/5z8/1k4/4/2/5e8657887c532.jpg)](mailto:?subject=Check%20out%20this%20story%20from%20Morning%20Brew%21&body=Halloween%20Pop-Up%20Shops%20Have%20Mixed%20Success%20In%202020%3A%20https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020%0A%0AWant%20more%20great%20content%3F%20Subscribe%20to%20Daily%20Brew%20%E2%80%94%20Delivering%20the%20latest%20business%20news%20from%20Wall%20St.%20to%20Silicon%20Valley%2C%20daily.%0Ahttps%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fr%2F%3Fkid%3D{{profile.vars.referral_code}}%26utm_source%3Demail_share%0A) | | |\n| --- |\n\n| |     |\n| --- |\n| ### FOOD'
matched = q_search.search(part)
matched.group()

'1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+)'